# Use Model Context Protocol (MCP) as tools with Strands Agent

## Overview
The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) is an open protocol that standardizes how applications provide context to Large Language Models (LLMs). Strands AI SDK integrates with MCP to extend agent capabilities through external tools and services.

MCP enables communication between agents and MCP servers that provide additional tools. The Strands Agent SDK includes built-in support for connecting to MCP servers and using their tools.

In this example we will show you how to use MCP tools on your Strands Agent. We will use the [AWS Documentation MCP server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server/) which provides tools to access AWS documentation, search for content, and get recommendations. This MCP server has 3 main features:

- **Read Documentation**: Fetch and convert AWS documentation pages to markdown format
- **Search Documentation**: Search AWS documentation using the official search API
- **Recommendations**: Get content recommendations for AWS documentation pages



## Agent Details
<div style="float: left; margin-right: 20px;">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|Feature used        |MCP Tools                                          |
|Agent Structure     |Single agent architecture                          |

</div>

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## Key Features
* **Single agent architecture**: this example creates a single agent that interacts with MCP tools
* **MCP tools**: Integration of MCP tools with your agent

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock

Let's now install the requirement packages for our Strands Agent agent

In [1]:
# installing pre-requisites
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.4/23.4 MB 57.3 MB/s  0:00:00m0:00:010:01


### Importing dependency packages

Now let's import the dependency packages

In [2]:
import threading
import time
from datetime import timedelta

from mcp import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamablehttp_client
from mcp.server import FastMCP
from strands import Agent
from strands.tools.mcp import MCPClient

### Connect to MCP server using stdio transport

[Transports](https://modelcontextprotocol.io/specification/2025-03-26/basic/transports) in MCP provide the foundations for communication between clients and servers. It handles the underlying mechanics of how messages are sent and received. At the moment there are three standards transport implementations built-in in MCP:

- **Standard Input/Output (stdio)**: enables communication through standard input and output streams. It is particularly useful for local integrations and command-line tools
- **Streamable HTTP**: this replaces the HTTP+SSE transport from previous protocol version. In the Streamable HTTP transport, the server operates as an independent process that can handle multiple client connections. This transport uses HTTP POST and GET requests. Server can optionally make use of Server-Sent Events (SSE) to stream multiple server messages. This permits basic MCP servers, as well as more feature-rich servers supporting streaming and server-to-client notifications and requests.
- **SSE**: legacy transport for HTTP-based MCP servers that use Server-Sent Events transport  

Overall, you should use stdio for building command-line tools, implementing local integrations and working with shell scripts. You should use Streamable HTTP transports when you need a flexible and efficient way for AI agents to communicate with tools and services, especially when dealing with stateless communication or when minimizing resource usage is crucial.

You can also use **custom transports** implementation for your specific needs. 


Let's now connect to the MCP server using stdio transport. First of all, we will use the class `MCPClient` to connect to the [AWS Documentation MCP Server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server/). This server provides tools to access AWS documentation, search for content, and get recommendations.

In [3]:
# Connect to an MCP server using stdio transport
stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `stdio_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [4]:
# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    tools = stdio_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = agent("What is Amazon Bedrock pricing model. Be concise.")


Tool #1: search_documentation

Tool #2: search_documentation

Tool #3: read_documentation

Tool #4: search_documentation

Tool #5: read_documentation
Based on the AWS documentation, here's Amazon Bedrock's pricing model:

## Amazon Bedrock Pricing Model

Amazon Bedrock offers **two main pricing models**:

### 1. **On-Demand (Pay-as-you-go)**
- Pay per **input and output token** processed
- No upfront commitments
- Pricing varies by model

### 2. **Provisioned Throughput**
- Purchase dedicated capacity measured in **Model Units (MUs)**
- Billed **hourly** at a fixed cost
- Pricing depends on:
  - Model selected
  - Number of Model Units
  - Commitment duration (no commitment, 1 month, or 6 months)
  - Longer commitments = greater discounts
- **Required** for custom models

**Note**: You can increase throughput using cross-Region inference profiles with both pricing models. Additional charges apply for features like Guardrails evaluation.

For detailed pricing, see the [Amazon Bedrock P

### Connect to MCP server using Streamable HTTP

Let's now connect to the MCP server using Streamable HTTP transport. First let's start a simple MCP server using Streamable HTTP transport. 

For this example we will create our own MCP server. The architecture will look as following

<div style="text-align:center">
    <img src="images/architecture_2.png" width="65%" />
</div>

In [5]:
# Create an MCP server
mcp = FastMCP("Calculator Server")

# Define a tool


@mcp.tool(description="Calculator tool which performs calculations")
def calculator(x: int, y: int) -> int:
    return x + y


@mcp.tool(description="This is a long running tool")
def long_running_tool(name: str) -> str:
    time.sleep(25)
    return f"Hello {name}"


def main():
    mcp.run(transport="streamable-http", mount_path="mcp")

Let's now start a thread with the `streamable-http` server

In [6]:
thread = threading.Thread(target=main)
thread.start()

INFO:     Started server process [6736]
INFO:     Waiting for application startup.


[03/07/26 18:07:12] INFO     StreamableHTTP session manager started                  ]8;id=863583;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=630997;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#115\115]8;;\

INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


#### Integrating Streamable HTTP client with Agent

Now let's use `streamablehttp_client` integrate this server with a simple agent. 

In [7]:
def create_streamable_http_transport():
    return streamablehttp_client("http://localhost:8000/mcp")


streamable_http_mcp_client = MCPClient(create_streamable_http_transport)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `streamable_http_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [8]:
with streamable_http_mcp_client:
    tools = streamable_http_mcp_client.list_tools_sync()

    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = str(agent("What is 2 + 2?"))

[03/07/26 18:07:30] INFO     Created new transport with session ID:                  ]8;id=41841;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=692791;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#239\239]8;;\
                             b40f9e08d05e42c58ef28ed5928fbe34                                                      

INFO:     127.0.0.1:36540 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=175996;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=52767;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: b40f9e08d05e42c58ef28ed5928fbe34           ]8;id=871987;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=358526;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=960747;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=250595;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

2 + 2 = 4

[03/07/26 18:07:34] INFO     Terminating session: b40f9e08d05e42c58ef28ed5928fbe34           ]8;id=572821;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=262089;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py#779\779]8;;\

INFO:     127.0.0.1:36576 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=124406;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=49520;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     GET stream disconnected, reconnecting in 1000ms...              ]8;id=397728;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=260529;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#298\298]8;;\

### Direct Tool Invocation

While tools are typically invoked by the agent based on user requests, you can also call MCP tools directly. This can be useful for workflow scenarios where you orchestrate multiple tools together.

In [9]:
query = {"x": 10, "y": 20}

with streamable_http_mcp_client:
    # direct tool invocation
    result = streamable_http_mcp_client.call_tool_sync(
        tool_use_id="tool-123", name="calculator", arguments=query
    )

    # Process the result
    print(f"Calculation result: {result['content'][0]['text']}")

[03/07/26 18:07:39] INFO     Created new transport with session ID:                  ]8;id=996391;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=103625;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#239\239]8;;\
                             3581ae71197949668989a9c831772660                                                      

INFO:     127.0.0.1:36592 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=318791;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=650135;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 3581ae71197949668989a9c831772660           ]8;id=228193;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=504296;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=948552;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=58163;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

                    INFO     Processing request of type ListToolsRequest                              ]8;id=969278;file:///opt/conda/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=108415;file:///opt/conda/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#713\713]8;;\

Calculation result: 30


                    INFO     Terminating session: 3581ae71197949668989a9c831772660           ]8;id=665992;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=559325;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py#779\779]8;;\

INFO:     127.0.0.1:36634 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=635210;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=889812;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

You can optionally also provide `read_timeout_seconds` while calling an MCP server tool to avoid it running for too long

In [10]:
with streamable_http_mcp_client:
    try:
        result = streamable_http_mcp_client.call_tool_sync(
            tool_use_id="tool-123",
            name="long_running_tool",
            arguments={"name": "Amazon"},
            read_timeout_seconds=timedelta(seconds=30),
        )

        if result["status"] == "error":
            print(f"Tool execution failed: {result['content'][0]['text']}")
        else:
            print(f"Tool execution succeeded: {result['content'][0]['text']}")
    except Exception as e:
        print(f"Tool call timed out or failed: {str(e)}")

[03/07/26 18:07:44] INFO     Created new transport with session ID:                  ]8;id=932758;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=94370;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#239\239]8;;\
                             09e1d73fa415417999b8c63a706097a2                                                      

INFO:     127.0.0.1:48070 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=71729;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=466883;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 09e1d73fa415417999b8c63a706097a2           ]8;id=272250;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=681402;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=745049;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=788447;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

Tool execution succeeded: Hello Amazon


                    INFO     Terminating session: 09e1d73fa415417999b8c63a706097a2           ]8;id=935654;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=635172;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py#779\779]8;;\

INFO:     127.0.0.1:40624 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=833764;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=269483;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     GET stream disconnected, reconnecting in 1000ms...              ]8;id=765246;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=808053;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#298\298]8;;\

### Interacting with multiple MCP servers

With Strands Agents you can also interact with multiple MCP servers using the same agent and configure tools setups such as the max number of tools that can be used in parallel (`max_parallel_tools`). Let's create a new agent to showcase this configuration:

<div style="text-align:center">
    <img src="images/architecture_3.png" width="85%" />
</div>

In this agent, we will again use the AWS Documentation MCP server and we will also use the [AWS CDK MCP Server](https://awslabs.github.io/mcp/servers/cdk-mcp-server/) which helps with AWS Cloud Development Kit (CDK) best practices, infrastructure as code patterns and security compliance with CDK Nag.

First let's connect to the two MCP servers using the stdio transport

In [11]:
# Connect to an MCP server using stdio transport
aws_docs_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

# Connect to an MCP server using stdio transport
cdk_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(command="uvx", args=["awslabs.cdk-mcp-server@latest"])
    )
)

#### Create Agent with MCP servers

Next we will create the agent with the tools from both MCP servers

In [12]:
# Create an agent with MCP tools
with aws_docs_mcp_client, cdk_mcp_client:
    # Get the tools from the MCP server
    tools = aws_docs_mcp_client.list_tools_sync() + cdk_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = agent(
        "What is Amazon Bedrock pricing model. Be concise. Also what are the best practices related to CDK?"
    )

I'll help you find information about Amazon Bedrock pricing and CDK best practices.
Tool #1: search_documentation

Tool #2: CDKGeneralGuidance
Let me get the actual Bedrock pricing documentation:
Tool #3: read_documentation
Let me search for more specific pricing information:
Tool #4: search_documentation

Tool #5: read_documentation
Perfect! Now I have the information needed. Let me provide a concise answer about both topics.

## **Amazon Bedrock Pricing Model**

Amazon Bedrock uses a **pay-per-use pricing model**:

- **On-Demand Inference** (default): Charged based on the number of **input and output tokens** processed
- **Serverless & auto-scaling**: No capacity planning or provisioning required
- **No upfront costs**: Pay only for what you use
- Additional charges for:
  - **Custom models**: Based on model copies, Custom Model Units (CMUs), and training time
  - **Guardrails**: Charges for each evaluation
  - **Provisioned Throughput**: Available for predictable, high-volume worklo

### Congratulations!

In this notebook you learned how to connect with MCP servers using Strands Agent and two MCP transport protocols: stdio and Streamable HTTP. You also learned how to connect multiple MCP servers to the same agent. Next, let's see how to use different models with your agent